In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

In [2]:
class AgentState(TypedDict):
    number1: int 
    operation: str 
    number2: int
    finalNumber: int

In [ ]:
def adder(state:AgentState) -> AgentState:
    """This node adds the 2 numbers"""
    state["finalNumber"] = state["number1"] + state["number2"]

    return state

def subtractor(state:AgentState) -> AgentState:
    """This node subtracts the 2 numbers"""
    state["finalNumber"] = state["number1"] - state["number2"]
    return state

#adding a router node which will decide next node
def decide_next_node(state:AgentState) -> AgentState:
    """This node will select the next node of the graph"""

    if state["operation"] == "+":
        return "addition_operation"
    
    elif state["operation"] == "-":
        return "subtraction_operation" 

In [ ]:
graph = StateGraph(AgentState)

graph.add_node("add_node", adder)
graph.add_node("subtract_node", subtractor)
graph.add_node("router", lambda state:state) # passthrough function ---here it represents input state is the output state
#we are using this passthrough function because in the decide next node we are not changing the state since we are comparing 
#two operators not assigning them so the state will remain as it is after it moves through state

graph.add_edge(START, "router") 

graph.add_conditional_edges(
    "router", #source part
    decide_next_node, #path of function which we are implementing as condition
#defining a path map
    {
        # Edge: Nodes for both the operation
        "addition_operation": "add_node",
        "subtraction_operation": "subtract_node"
    }

)

graph.add_edge("add_node", END)
graph.add_edge("subtract_node", END)

app = graph.compile()

In [5]:
#Visualizing this agent code
from IPython.display import Markdown, display
display(Markdown(f"```mermaid\n{app.get_graph().draw_mermaid()}\n```"))

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	add_node(add_node)
	subtract_node(subtract_node)
	router(router)
	__end__([<p>__end__</p>]):::last
	__start__ --> router;
	router -. &nbsp;addition_operation&nbsp; .-> add_node;
	router -. &nbsp;subtraction_operation&nbsp; .-> subtract_node;
	add_node --> __end__;
	subtract_node --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [6]:
initial_state_1 = AgentState(number1 = 10, operation="-", number2 = 5)
print(app.invoke(initial_state_1))

{'number1': 10, 'operation': '-', 'number2': 5, 'finalNumber': 5}


In [7]:
# This way still works!

result = app.invoke({"number1": 10, "operation": "-", "number2": 5})
print(result)

{'number1': 10, 'operation': '-', 'number2': 5, 'finalNumber': 5}
